In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

In [2]:
from google.colab import files
uploaded = files.upload()


Saving phishing_email_dataset.xlsx to phishing_email_dataset.xlsx


In [3]:
df = pd.read_excel("phishing_email_dataset.xlsx")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (500, 10)


,Email_ID,Subject,Body,URL,Contains_Suspicious_Keywords,URL_Length,Has_IP_in_URL,Num_Links,Has_Attachment,Label
0,1,Urgent action required,Click here to update payment Your account will...,https://secure-login-alert.com/mzxcowub,1,71,1,1,1,Phishing
1,2,Meeting scheduled for tomorrow,Invoice attached for your review Meeting sched...,https://example.com/nnsaj,0,26,0,2,1,Safe
2,3,Invoice attached for your review,Your order has been shipped Your order has bee...,https://service.org/lgyxs,0,73,0,2,1,Safe
3,4,Urgent action required,Urgent action required Click here to update pa...,https://verify-account-now.net/dydgbsad,1,53,1,4,0,Phishing
4,5,Your account will be suspended,Verify your account immediately Click here to ...,http://security-check.xyz/oymukolk,1,70,1,1,0,Phishing


In [4]:
df["text"] = df["Subject"] + " " + df["Body"]

In [5]:
df["Label"] = df["Label"].map({"Safe": 0, "Phishing": 1})


In [6]:
vectorizer = TfidfVectorizer(max_features=3000)
X_text = vectorizer.fit_transform(df["text"]).toarray()

In [7]:
X_num = df[[
    "Contains_Suspicious_Keywords",
    "URL_Length",
    "Has_IP_in_URL",
    "Num_Links",
    "Has_Attachment"
]].values

In [14]:
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

In [15]:
X = np.hstack((X_text, X_num_scaled))
y = df["Label"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [11]:
y_pred = model.predict(X_test)

In [12]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Confusion Matrix:
[[54  0]
 [ 0 46]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        54
           1       1.00      1.00      1.00        46

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [16]:
# Install joblib (if not already)
!pip install joblib

import joblib

# Save trained model
joblib.dump(model, "phishing_model.pkl")

# Save vectorizer (for text processing)
joblib.dump(vectorizer, "vectorizer.pkl")

# Save scaler (for numerical features)
joblib.dump(scaler, "scaler.pkl")

print("All files saved successfully!")

All files saved successfully!
